# Part 5 - Guardrail Pipeline

Demonstrate input filter + calibrated model + human review queue.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from assignment_workflow import load_and_split_data, tokenize_df, fit_probability_calibrator, evaluate_bands
from pipeline import ModerationPipeline, count_filter_hits

_, eval_df = load_and_split_data('jigsaw-unintended-bias-train.csv')
sample_df = eval_df.sample(n=1000, random_state=42).reset_index(drop=True)
model_dir = 'saved_model/part1_baseline'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
trainer = Trainer(model=model, args=TrainingArguments(output_dir='tmp_part5', report_to=[]))
dataset = tokenize_df(sample_df, tokenizer)
prob = torch.softmax(torch.tensor(trainer.predict(dataset).predictions), dim=1).numpy()[:,1]
calibrator = fit_probability_calibrator(prob, sample_df['label'].values)
pipe = ModerationPipeline(model=model, tokenizer=tokenizer, calibrator=calibrator)

In [ ]:
decisions = [pipe.predict(t) for t in sample_df['comment_text'].fillna('').tolist()]
layers = pd.Series([d['layer'] for d in decisions]).value_counts(normalize=True)
print('Layer distribution:')
print(layers)
layers.plot(kind='bar', title='Decision Layer Distribution')
plt.show()

print('Input filter category counts:')
print(count_filter_hits(sample_df['comment_text'].fillna('').tolist()))

for low, high in [(0.45, 0.55), (0.4, 0.6), (0.3, 0.7)]:
    r = evaluate_bands(calibrator.predict_proba(prob.reshape(-1,1))[:,1], low, high)
    print(r)